In [1]:
import os, json, shutil, hashlib
import pandas as pd

os.chdir("/scratch/jq2uw/derm_vlms")

VLMS = {
    "DermatoLlama": "results/dermato_llama_predictions_paired.csv",
    "MedGemma":     "results/medgemma_predictions_paired.csv",
    "GPT-5.3":      "results/gpt53_predictions_paired.csv",
}
IMAGE_DIR     = "results/images"
INTERFACE_SRC = "res_eng/interface_simple"
OUTPUT_DIR    = "results/interface_share"
OUTPUT_HTML   = os.path.join(OUTPUT_DIR, "index_simple.html")
OUTPUT_IMGS   = os.path.join(OUTPUT_DIR, "images")

QUESTIONS = ["Is the AI’s assessment accurate for this case? If not, what is incorrect and what should it say instead?"]

# --- load CSV data, keep only combined images and describe_then_classify ---
vlm_data = {}
all_ids = set()
for name, path in VLMS.items():
    if not os.path.exists(path):
        print(f"[SKIP] {name}: {path} not found (predictions may still be running)")
        continue
    df = pd.read_csv(path)
    df = df[df["id"].str.endswith("_combined")]
    if df.empty:
        print(f"[SKIP] {name}: no combined rows yet")
        continue
    records = df[["id", "describe_then_classify"]].to_dict("records")
    vlm_data[name] = records
    all_ids.update(df["id"])

# --- Build image path map (relative to the HTML file) ---
images = {}
for img_id in sorted(all_ids):
    images[img_id] = f"images/{img_id}.jpg"

# --- Copy high-res images to output folder ---
os.makedirs(OUTPUT_IMGS, exist_ok=True)
copied = 0
for img_id in sorted(all_ids):
    src = os.path.join(IMAGE_DIR, f"{img_id}.jpg")
    dst = os.path.join(OUTPUT_IMGS, f"{img_id}.jpg")
    shutil.copy2(src, dst)
    copied += 1

fingerprint_src = json.dumps(QUESTIONS) + json.dumps(sorted(all_ids))
CONFIG_HASH = hashlib.md5(fingerprint_src.encode()).hexdigest()[:8]

print(f"Loaded {len(vlm_data)} VLMs, {len(all_ids)} images (config hash: {CONFIG_HASH})")
print(f"Copied {copied} images to {OUTPUT_IMGS}")

Loaded 3 VLMs, 10 images (config hash: feabceb5)
Copied 10 images to results/interface_share/images


In [2]:
# --- Read source files ---
css      = open(os.path.join(INTERFACE_SRC, "style.css")).read()
js       = open(os.path.join(INTERFACE_SRC, "app.js")).read()
template = open(os.path.join(INTERFACE_SRC, "template.html")).read()

# --- Build the data block injected before app.js ---
data_block = f"""
const VLM_DATA  = {json.dumps(vlm_data)};
const IMAGES    = {json.dumps(images)};
const QUESTIONS = {json.dumps(QUESTIONS)};
const TASK_KEY  = 'describe_then_classify';
const SK = 'vlm_eval_simple_{CONFIG_HASH}';
"""

# --- Stitch into single HTML ---
html = template
html = html.replace("__CSS__", css)
html = html.replace("__DATA__", data_block)
html = html.replace("__JS__", js)

os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(OUTPUT_HTML, "w") as f:
    f.write(html)

size_kb = os.path.getsize(OUTPUT_HTML) / 1024
print(f"Generated {OUTPUT_HTML} ({size_kb:.1f} KB)")
print(f"Output folder: {OUTPUT_DIR}/")
print(f"  index_simple.html + images/ -> ready to zip and share")

Generated results/interface_share/index_simple.html (57.9 KB)
Output folder: results/interface_share/
  index_simple.html + images/ -> ready to zip and share
